# Hallmark: private data

Catalog files from several places, download them through a local cache, and
share their checksums. Run the cells in order with Hallmark installed; no
server account is needed.

## Setup

All demo files go in a temporary folder. A small web server on this computer
plays the data server, with four sample files.

In [1]:
import os
import tempfile
from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from threading import Thread

from hallmark import Repo

workspace = tempfile.mkdtemp(prefix="hallmark-demo-")
os.chdir(workspace)
os.mkdir("server")
for name in ["M87_001.txt", "M87_002.txt", "SgrA_001.txt", "SgrA_002.txt"]:
    with open(f"server/{name}", "w") as file:
        file.write(f"demo data for {name}\n")

In [2]:
# Serve the folder in the background, without logging each request.
class QuietHandler(SimpleHTTPRequestHandler):
    def log_message(self, *args):
        pass


server = ThreadingHTTPServer(("127.0.0.1", 0), partial(QuietHandler, directory="server"))
Thread(target=server.serve_forever, daemon=True).start()
DATA_URL = f"http://127.0.0.1:{server.server_port}"
print("Using", DATA_URL)

Using http://127.0.0.1:32777


To use the team demo server instead, set
`DATA_URL = "ssh://hallmark-demo/srv/hallmark_demo_data"`
after connecting to it as described in [demo_cli.ipynb](demo_cli.ipynb).

In [3]:
eht = Repo.init("eht")
eht.set_identity("Hallmark Demo", "demo@example.invalid")

## Several sources

Each `add` of a URL records a *source*: a folder, a filename pattern and
filters. One catalog can hold several sources, from one server or many.

In [4]:
# M87 files, named "m87"; the pattern reads the day from each name.
eht.add(DATA_URL + "/M87_{day:03d}.txt", source="m87");

In [5]:
# Sgr A* files: a second source in the same folder.
eht.add(DATA_URL + "/SgrA_{day:03d}.txt", source="sgra");

### Keep sources apart

Downloads mirror each server, so two sources cannot catalog the same path.
`prefix` gives a source its own folder, for example for a second server with
the same file names. Here, a second copy of the M87 files goes to `mirror/`.

In [6]:
eht.add(DATA_URL, filter="M87_*", source="mirror", prefix="mirror");

The configuration lists the sources.

In [7]:
eht.state.config["sources"]

[{'name': 'm87', 'remote': 'origin', 'fmt': 'M87_{day:03d}.txt'},
 {'name': 'sgra', 'remote': 'origin', 'fmt': 'SgrA_{day:03d}.txt'},
 {'name': 'mirror',
  'remote': 'origin',
  'prefix': 'mirror/',
  'filter': ['M87_*']}]

Each catalog row names its source.

In [8]:
eht.state.data[["path", "source", "day"]]

,path,source,day
0,M87_001.txt,m87,1
1,M87_002.txt,m87,2
2,SgrA_001.txt,sgra,1
3,SgrA_002.txt,sgra,2
4,mirror/M87_001.txt,mirror,
5,mirror/M87_002.txt,mirror,


In [9]:
eht.commit("Catalog M87 and Sgr A*")

True

## Download through the cache

`sources` selects the files of one source.

In [10]:
print(eht.plan_download(sources=["sgra"]).summary())

2 file(s); 0 bytes known; 2 file(s) with unknown size; estimated duration: unknown
Source: http://127.0.0.1:32777/
Backend: http
Destination: /tmp/hallmark-demo-bve9w2yl/eht


Each file goes to `.hm/cache` first and is checked, then linked into the
folder. A failed or interrupted download leaves no partial file behind.

In [11]:
eht.download(eht.plan_download(all_files=True), approved=True)

{'succeeded': 6, 'failed': 0, 'total_bytes': 158, 'errors': [], 'staged': 6}

Downloaded files are read-only links to the cache, so the cached copy cannot
change by accident. The two `M87_001.txt` copies share one cached file.

In [12]:
for path in ["M87_001.txt", "mirror/M87_001.txt"]:
    info = os.stat(eht.worktree / path)
    print(path, "mode", oct(info.st_mode & 0o777), "links", info.st_nlink)

M87_001.txt mode 0o400 links 3
mirror/M87_001.txt mode 0o400 links 3


### Record the checksums

The download staged each file's SHA-1. Commit it so later downloads, including
your collaborators', are checked against it.

In [13]:
eht.status()["staged"]["modified"]

['M87_001.txt',
 'M87_002.txt',
 'SgrA_001.txt',
 'SgrA_002.txt',
 'mirror/M87_001.txt',
 'mirror/M87_002.txt']

In [14]:
eht.commit("Record downloaded checksums")

True

### Restore a file from the cache

A deleted file comes back from the cache without a transfer.

In [15]:
(eht.worktree / "SgrA_001.txt").unlink()
plan = eht.plan_download(all_files=True)
print(plan.summary())

1 file(s); 0 bytes; estimated duration: unknown
Source: http://127.0.0.1:32777/
Backend: http
Destination: /tmp/hallmark-demo-bve9w2yl/eht
From the local cache: 1 file(s) (no transfer)
Already downloaded: M87_001.txt
Already downloaded: M87_002.txt
Already downloaded: SgrA_002.txt
Already downloaded: mirror/M87_001.txt
Already downloaded: mirror/M87_002.txt


In [16]:
eht.download(plan, approved=True)

{'succeeded': 1,
 'failed': 0,
 'total_bytes': 0,
 'errors': [],
 'from_cache': ['SgrA_001.txt']}

## Other servers

The same calls work with WebDAV, S3 and other SSH servers. These need your own
accounts, so they are not run here.

```python
# WebDAV, such as signed-in CyVerse
repo.add("webdavs://data.cyverse.org/dav/iplant/home/alice/eht/", auth="cyverse")
# S3-compatible storage (pip install hallmark[s3])
repo.add("s3://ehtc-sims/m87/", auth="osn",
         backend_options={"endpoint_url": "https://sdsc.osn.xsede.org"})
# Let an SSH server compute checksums that are not published
repo.add("ssh://lab-data/srv/export/", server_checksum="sha256")
```

### Credentials

`auth="NAME"` refers to a profile in `~/.config/hallmark/auth.yml`. A profile
says where a secret comes from and never stores it:

```yaml
version: 2
profiles:
  cyverse:
    type: http
    hosts: [data.cyverse.org]
    scheme: basic
    username: alice
    password: {env: CYVERSE_PASSWORD}
```

## Clean up

Stop the server and remove the temporary demo files.

In [17]:
import shutil

server.shutdown()
os.chdir(tempfile.gettempdir())
shutil.rmtree(workspace)